In [110]:
import psycopg
import keyring
import yaml
import re
import os


In [113]:
TYPE_MAP = {
    "character varying": "string",
    "text": "string",
    "integer": "int",
    "bigint": "int",
    "boolean": "bool",
    "timestamp without time zone": "timestamp",
    "timestamp with time zone": "timestamptz",
}


def normalize_type(data_type: str) -> str:
    base = re.sub(r"\(.*\)", "", data_type).strip()
    return TYPE_MAP.get(base, base)


def prettify_name(name: str) -> str:
    clean = name.strip().lower()

    special = {
        "id": "Unique identifier",
        "created_at": "Timestamp when the record was created",
        "updated_at": "Timestamp when the record was last updated",
        "deleted_at": "Timestamp when the record was deleted",
        "created_by": "User that created the record",
        "updated_by": "User that last updated the record",
        "deleted_by": "User that deleted the record",
        "is_active": "Whether the record is active",
        "is_deleted": "Whether the record is deleted",
        "status": "Current status",
        "type": "Classification or type",
        "name": "Display name",
        "email": "Email address",
    }

    if clean in special:
        return special[clean]

    if clean.endswith("_id"):
        entity = clean[:-3].replace("_", " ")
        return f"Identifier for the associated {entity}"

    if clean.startswith("is_"):
        return f"Whether {clean[3:].replace('_', ' ')}"

    if clean.startswith("has_"):
        return f"Whether the record has {clean[4:].replace('_', ' ')}"

    if clean.endswith("_at"):
        return f"Timestamp when {clean[:-3].replace('_', ' ')}"

    if clean.endswith("_date"):
        return f"Date for {clean[:-5].replace('_', ' ')}"

    return clean.replace("_", " ").capitalize()


def prettify_table_name(table: str) -> str:
    clean = table.strip().lower().replace("_", " ")

    if clean.endswith("s"):
        return f"Records representing {clean}"

    return f"Records for {clean}"


def enrich_table_description(table: str, foreign_keys: list[dict]) -> str:
    description = prettify_table_name(table)

    if not foreign_keys:
        return description

    referenced_tables = sorted({
        fk["references"]["table"]
        for fk in foreign_keys
        if fk.get("references", {}).get("table")
    })

    if referenced_tables:
        refs = ", ".join(referenced_tables)
        description += f"; related to {refs}"

    return description


def infer_semantic(column_name: str, data_type: str, is_primary_key: bool):
    name = column_name.lower()
    t = data_type.lower()
    col = column_name

    numeric_types = {
        "smallint",
        "integer",
        "bigint",
        "numeric",
        "decimal",
        "real",
        "double precision",
        "money",
    }

    date_types = {
        "date",
        "timestamp",
        "timestamp without time zone",
        "timestamp with time zone",
    }

    # ✅ Primary key → total row count
    if is_primary_key:
        return {
            "role": "dimension",
            "dimension_type": "identifier",
            "example_measures": [
                {
                    "name": "total_count",
                    "sql": f"COUNT({col})",
                    "description": "Total number of records",
                }
            ],
        }

    # ✅ Foreign key / ID
    if name.endswith("_id"):
        entity = name[:-3].replace("_", " ")
        return {
            "role": "dimension",
            "dimension_type": "identifier",
            "example_measures": [
                {
                    "name": f"distinct_{name}_count",
                    "sql": f"COUNT(DISTINCT {col})",
                    "description": f"Number of distinct {entity} values",
                }
            ],
        }

    # ✅ Time fields
    if t in date_types or name.endswith("_at") or name.endswith("_date"):
        return {
            "role": "dimension",
            "dimension_type": "time",
            "example_measures": [
                {
                    "name": f"min_{name}",
                    "sql": f"MIN({col})",
                    "description": f"Earliest {name.replace('_', ' ')}",
                },
                {
                    "name": f"max_{name}",
                    "sql": f"MAX({col})",
                    "description": f"Latest {name.replace('_', ' ')}",
                },
            ],
        }

    # ✅ Boolean
    if t == "boolean" or name.startswith(("is_", "has_")):
        subject = name.replace("is_", "").replace("has_", "").replace("_", " ")
        return {
            "role": "dimension",
            "dimension_type": "boolean",
            "example_measures": [
                {
                    "name": f"true_{name}_count",
                    "sql": f"SUM(CASE WHEN {col} THEN 1 ELSE 0 END)",
                    "description": f"Count of records where {subject} is true",
                }
            ],
        }

    # ✅ Numeric
    if t in numeric_types:
        measures = [
            {
                "name": f"avg_{name}",
                "sql": f"AVG({col})",
                "description": f"Average {name.replace('_', ' ')}",
            }
        ]

        if any(
            k in name for k in ["amount", "total", "price", "cost", "revenue", "sales"]
        ):
            measures.insert(
                0,
                {
                    "name": f"total_{name}",
                    "sql": f"SUM({col})",
                    "description": f"Total {name.replace('_', ' ')}",
                },
            )

        if any(k in name for k in ["count", "quantity", "qty", "num"]):
            measures = [
                {
                    "name": f"total_{name}",
                    "sql": f"SUM({col})",
                    "description": f"Total {name.replace('_', ' ')}",
                }
            ]

        return {"role": "measure", "example_measures": measures}

    # ✅ Default categorical
    return {"role": "dimension", "dimension_type": "categorical"}


def get_table_schema_yaml(conn, table, schema="public"):
    query = """
    SELECT
        a.attname AS column_name,
        pg_catalog.format_type(a.atttypid, a.atttypmod) AS data_type,
        NOT a.attnotnull AS nullable,
        pg_get_expr(d.adbin, d.adrelid) AS default_value,

        con.conname AS constraint_name,
        con.contype AS constraint_type,

        ns_ref.nspname AS foreign_schema,
        cls_ref.relname AS foreign_table,
        att_ref.attname AS foreign_column

    FROM pg_attribute a
    JOIN pg_class cls
        ON cls.oid = a.attrelid
    JOIN pg_namespace ns
        ON ns.oid = cls.relnamespace

    LEFT JOIN pg_attrdef d
        ON d.adrelid = a.attrelid
       AND d.adnum = a.attnum

    LEFT JOIN pg_constraint con
        ON con.conrelid = a.attrelid
       AND a.attnum = ANY(con.conkey)

    LEFT JOIN pg_class cls_ref
        ON cls_ref.oid = con.confrelid

    LEFT JOIN pg_namespace ns_ref
        ON ns_ref.oid = cls_ref.relnamespace

    LEFT JOIN pg_attribute att_ref
        ON att_ref.attrelid = con.confrelid
       AND att_ref.attnum = con.confkey[
            array_position(con.conkey, a.attnum)
       ]

    WHERE ns.nspname = %s
      AND cls.relname = %s
      AND a.attnum > 0
      AND NOT a.attisdropped

    ORDER BY a.attnum, con.contype;
    """

    with conn.cursor() as cur:
        cur.execute(query, (schema, table))
        rows = cur.fetchall()

    output = {
        "schema": schema,
        "table": table,
        "primary_keys": [],
        "columns": {},
    }

    table_measures = []

    foreign_keys = []
    seen_foreign_keys = set()

    for (
        column_name,
        data_type,
        nullable,
        default_value,
        constraint_name,
        constraint_type,
        foreign_schema,
        foreign_table,
        foreign_column,
    ) in rows:
        if column_name not in output["columns"]:
            normalized_type = normalize_type(data_type)

            semantic = infer_semantic(
                column_name, normalized_type, column_name in output["primary_keys"]
            )

            for measure in semantic.pop("example_measures", []):
                table_measures.append(measure)

            output["columns"][column_name] = {
                "type": normalized_type,
                "nullable": nullable,
                "description": prettify_name(column_name),
                "semantic": semantic,
            }

        if constraint_type == "p" and column_name not in output["primary_keys"]:
            output["primary_keys"].append(column_name)

        elif constraint_type == "f":
            fk_key = (
                constraint_name,
                column_name,
                foreign_schema,
                foreign_table,
                foreign_column,
            )

            if fk_key not in seen_foreign_keys:
                foreign_keys.append({
                    "column": column_name,
                    "references": {
                        "schema": foreign_schema,
                        "table": foreign_table,
                        "column": foreign_column,
                    },
                })
                seen_foreign_keys.add(fk_key)

    description = enrich_table_description(table, foreign_keys)

    if foreign_keys:
        output = {
            "schema": output["schema"],
            "table": output["table"],
            "description": description,
            "primary_keys": output["primary_keys"],
            "foreign_keys": foreign_keys,
            "example_measures": table_measures,
            "columns": output["columns"],
        }
    else:
        output = {
            "schema": output["schema"],
            "table": output["table"],
            "description": description,
            "primary_keys": output["primary_keys"],
            "example_measures": table_measures,
            "columns": output["columns"],
        }

    return yaml.safe_dump(output, sort_keys=False)


def get_tables(conn, schema="public"):
    with conn.cursor() as cur:
        cur.execute(
            """
        SELECT tablename
        FROM pg_catalog.pg_tables
        WHERE schemaname = %s;
        """,
            (schema,),
        )
        return [row[0] for row in cur.fetchall()]


def export_all_tables(conn, output_dir="schemas", schema="public"):
    os.makedirs(output_dir, exist_ok=True)

    tables = get_tables(conn, schema)

    for table in tables:
        yaml_str = get_table_schema_yaml(conn, table, schema)

        filename = os.path.join(output_dir, f"{table}.yaml")

        with open(filename, "w", encoding="utf-8") as f:
            f.write(yaml_str)

        print(f"Wrote {filename}")

In [114]:
user = "students"
conn = psycopg.connect(
    host="dpg-d35pib0dl3ps7394mc4g-a.oregon-postgres.render.com",
    port=5432,
    user=user,
    password=keyring.get_password("futureproof_ds_db", user),
    dbname="beam_neb0",
)
export_all_tables(conn)

Wrote schemas\users.yaml
Wrote schemas\subscriptions.yaml
Wrote schemas\sessions.yaml
Wrote schemas\payments.yaml
